In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_medication_dispensed;
SET search_path TO qdp_medication_dispensed;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.medication_dispensed',
  '0.1',
  'medication_dispensed.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_medication_dispensed table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_medication_dispensed CASCADE;

CREATE TABLE hub_medication_dispensed (
    medication_dispensed_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    medication_dispensed_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_medication_dispensed ADD CONSTRAINT pk_meddispensed_hubmeddispensed_meddispensedid PRIMARY KEY (medication_dispensed_id);

COMMENT ON COLUMN hub_medication_dispensed.medication_dispensed_id IS 'Medication Dispensed Identity';




-- -----------------------------------------------------------
-- create sat_medication_dispensed table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_dispensed;

CREATE TABLE sat_medication_dispensed (
    sat_medication_dispensed_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    medication_dispensed_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	medication_prescription_id BIGINT,
	medication_request_id BIGINT,
	status_code VARCHAR,
	status_concept_id BIGINT,
	status_raw_code VARCHAR,
	status_raw_concept_id BIGINT,
	not_performed_reason_code VARCHAR,
	not_performed_reason_concept_id BIGINT,
	not_performed_reason_raw_code VARCHAR,
	not_performed_reason_raw_concept_id BIGINT,
	status_changed TIMESTAMP,
	medication_code VARCHAR,
	medication_concept_id BIGINT,
	medication_raw_code VARCHAR,
	medication_raw_concept_id BIGINT,
	location_id BIGINT,
	type_code VARCHAR,
	type_concept_id BIGINT,
	type_raw_code VARCHAR,
	type_raw_concept_id BIGINT,
	quantity INT,
	days_supply INT,
	recorded TIMESTAMP,
	when_prepared TIMESTAMP,
	when_handed_over TIMESTAMP,
	destination_location_id BIGINT,
	rendered_dosage_instructions TEXT,
	was_substituted BOOLEAN,
	substitution_type_code VARCHAR,
	substitution_type_concept_id BIGINT,
	substitution_type_raw_code VARCHAR,
	substitution_type_raw_concept_id BIGINT,
	substitution_responsible_party_id BIGINT NOT NULL,
	record_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_dispensed ADD CONSTRAINT pk_meddispensed_satmeddispensed_satmeddispensedid PRIMARY KEY (sat_medication_dispensed_id);
ALTER TABLE sat_medication_dispensed ADD CONSTRAINT fk_meddispensed_satmeddispensed_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_medication_dispensed.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_medication_dispensed.sat_medication_dispensed_id IS 'Medication Dispensed Identity Sat';
COMMENT ON COLUMN sat_medication_dispensed.medication_prescription_id IS 'Medication Prescription Identity. Cross-schema reference to qdp_medication_prescriptions';
COMMENT ON COLUMN sat_medication_dispensed.medication_request_id IS 'Unique Identifier for the Medication Line on a Prescription. Cross-schema reference to qdp_medication_requests';
COMMENT ON COLUMN sat_medication_dispensed.status_code IS 'Medication Dispensed Status Code. E.g. preparation, in-progress, cancelled, on-hold, completed, entered-in-error, stopped, declined, unknown. See https://hl7.org/fhir/R5/valueset-medicationdispense-status.html';
COMMENT ON COLUMN sat_medication_dispensed.not_performed_reason_code IS 'Why a dispense was not performed. E.g. frr01 (Order Stopped), frr02 (Stale-dated Order), frr03 (Incomplete data), frr04 (Product unavailable), frr05 (Ethical/religious), altchoice, clarif, drughigh, hospadm, outofstock, offmarket. See https://hl7.org/fhir/R5/valueset-medicationdispense-status-reason.html';
COMMENT ON COLUMN sat_medication_dispensed.status_changed IS 'When the status changed';
COMMENT ON COLUMN sat_medication_dispensed.medication_code IS 'What medication was supplied by this Dispense (SNOMED CT coded). See https://build.fhir.org/valueset-medication-codes.html';
COMMENT ON COLUMN sat_medication_dispensed.location_id IS 'The principal physical location where the dispense was performed';
COMMENT ON COLUMN sat_medication_dispensed.type_code IS 'The Type of Dispense. E.g. Trial fill, partial fill, emergency fill. See https://hl7.org/fhir/R5/valueset-medicationdispense-performer-function.html';
COMMENT ON COLUMN sat_medication_dispensed.quantity IS 'Amount dispensed. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.quantity';
COMMENT ON COLUMN sat_medication_dispensed.days_supply IS 'Amount of medication expressed as a timing amount. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.daysSupply';
COMMENT ON COLUMN sat_medication_dispensed.recorded IS 'When the recording of the dispense started';
COMMENT ON COLUMN sat_medication_dispensed.when_prepared IS 'When product was packaged and reviewed. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.whenPrepared';
COMMENT ON COLUMN sat_medication_dispensed.when_handed_over IS 'When product was given out. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.whenHandedOver';
COMMENT ON COLUMN sat_medication_dispensed.destination_location_id IS 'Where the medication was/will be sent';
COMMENT ON COLUMN sat_medication_dispensed.rendered_dosage_instructions IS 'Full representation of the dosage instructions in CommonMark GFM format';
COMMENT ON COLUMN sat_medication_dispensed.was_substituted IS 'Whether a substitution was or was not performed on the dispense';
COMMENT ON COLUMN sat_medication_dispensed.substitution_type_code IS 'Code signifying whether a different drug was dispensed from what was prescribed. E.g. E (equivalent), EC (equivalent composition), BC (brand composition), G (generic composition), TE (therapeutic alternative), F (formulary), N (none). See https://terminology.hl7.org/5.4.0/ValueSet-v3-ActSubstanceAdminSubstitutionCode.html';
COMMENT ON COLUMN sat_medication_dispensed.substitution_responsible_party_id IS 'Who is responsible for the substitution. E.g. Practitioner, PractitionerRole, Organization';
COMMENT ON COLUMN sat_medication_dispensed.record_id IS 'A Unique Identifier for a Data Record within the System that allows Data Ownership, Ownership History, Dynamic Attributes and Metadata to be associated with that Record';



-- -----------------------------------------------------------
-- create sat_medication_dispensed_dosage table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_dispensed_dosage;

CREATE TABLE sat_medication_dispensed_dosage (
	medication_dispensed_dosage_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	dosage_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_dispensed_dosage ADD CONSTRAINT pk_meddispensed_satmeddispenseddosage_meddispenseddosageid PRIMARY KEY (medication_dispensed_dosage_id);
ALTER TABLE sat_medication_dispensed_dosage ADD CONSTRAINT fk_meddispensed_satmeddispenseddosage_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_medication_dispensed_dosage.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_medication_dispensed_dosage.medication_dispensed_dosage_id IS 'Medication Dispensed Dosage Identity';
COMMENT ON COLUMN sat_medication_dispensed_dosage.dosage_id IS 'Unique Identifier for the Dosage. See https://build.fhir.org/dosage.html#Dosage';



-- -----------------------------------------------------------
-- create sat_dispensed_authorizing_prescription table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_authorizing_prescription;

CREATE TABLE sat_dispensed_authorizing_prescription (
	dispensed_authorizing_prescription_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	medication_request_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_authorizing_prescription ADD CONSTRAINT pk_meddispensed_satdispensedauthprescription_dispensedauthprescriptionid PRIMARY KEY (dispensed_authorizing_prescription_id);
ALTER TABLE sat_dispensed_authorizing_prescription ADD CONSTRAINT fk_meddispensed_satdispensedauthprescription_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_authorizing_prescription.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_authorizing_prescription.dispensed_authorizing_prescription_id IS 'Dispensed Authorizing Prescription Identity';
COMMENT ON COLUMN sat_dispensed_authorizing_prescription.medication_request_id IS 'Medication order that authorizes the dispense. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.authorizingPrescription';
COMMENT ON COLUMN sat_dispensed_authorizing_prescription.rank IS 'The recommended order the Dispense Authorizing Prescription Entries should be presented';



-- -----------------------------------------------------------
-- create sat_dispensed_based_on table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_based_on;

CREATE TABLE sat_dispensed_based_on (
	dispensed_based_on_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	care_plan_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_based_on ADD CONSTRAINT pk_meddispensed_satdispensedbasedon_dispensedbasedonid PRIMARY KEY (dispensed_based_on_id);
ALTER TABLE sat_dispensed_based_on ADD CONSTRAINT fk_meddispensed_satdispensedbasedon_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_based_on.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_based_on.dispensed_based_on_id IS 'Dispensed Based On Identity';
COMMENT ON COLUMN sat_dispensed_based_on.care_plan_id IS 'Plan that is fulfilled by this dispense. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.basedOn';
COMMENT ON COLUMN sat_dispensed_based_on.rank IS 'The recommended order the Dispense Based On Entries should be presented';



-- -----------------------------------------------------------
-- create sat_dispensed_category table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_category;

CREATE TABLE sat_dispensed_category (
	dispensed_category_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	category_code VARCHAR,
	category_concept_id BIGINT,
	category_raw_code VARCHAR,
	category_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_category ADD CONSTRAINT pk_meddispensed_satdispensedcategory_dispensedcategoryid PRIMARY KEY (dispensed_category_id);
ALTER TABLE sat_dispensed_category ADD CONSTRAINT fk_meddispensed_satdispensedcategory_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_category.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_category.dispensed_category_id IS 'Dispensed Category Identity';
COMMENT ON COLUMN sat_dispensed_category.category_code IS 'Type of medication dispense. E.g. inpatient, outpatient, community. See https://hl7.org/fhir/R5/valueset-medicationdispense-admin-location.html';
COMMENT ON COLUMN sat_dispensed_category.rank IS 'The recommended order the Dispense Category Codes should be presented';



-- -----------------------------------------------------------
-- create sat_dispensed_notes table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_notes;

CREATE TABLE sat_dispensed_notes (
	dispensed_note_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	author_reference_party_id BIGINT,
	author_string VARCHAR,
	time TIMESTAMP,
	text TEXT NOT NULL,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_notes ADD CONSTRAINT pk_meddispensed_satdispensednotes_dispensednoteid PRIMARY KEY (dispensed_note_id);
ALTER TABLE sat_dispensed_notes ADD CONSTRAINT fk_meddispensed_satdispensednotes_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_notes.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_notes.dispensed_note_id IS 'Dispensed Note Identity';
COMMENT ON COLUMN sat_dispensed_notes.author_reference_party_id IS 'The clinical Party Identifier that recorded this information. E.g. Practitioner, PractitionerRole, Patient, RelatedPerson, Organization. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.note';
COMMENT ON COLUMN sat_dispensed_notes.author_string IS 'The Individual that recorded this information';
COMMENT ON COLUMN sat_dispensed_notes.time IS 'When this information was recorded';
COMMENT ON COLUMN sat_dispensed_notes.text IS 'Clinician entered additional information about the Dispense';



-- -----------------------------------------------------------
-- create sat_dispensed_part_of table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_part_of;

CREATE TABLE sat_dispensed_part_of (
	dispensed_part_of_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	procedure_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_part_of ADD CONSTRAINT pk_meddispensed_satdispensedpartof_dispensedpartofid PRIMARY KEY (dispensed_part_of_id);
ALTER TABLE sat_dispensed_part_of ADD CONSTRAINT fk_meddispensed_satdispensedpartof_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_part_of.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_part_of.dispensed_part_of_id IS 'Dispensed Part Of Identity';
COMMENT ON COLUMN sat_dispensed_part_of.procedure_id IS 'Event that dispense is part of. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.partOf';
COMMENT ON COLUMN sat_dispensed_part_of.rank IS 'The recommended order the Dispense Part Of Entries should be presented';



-- -----------------------------------------------------------
-- create sat_dispensed_performer table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_performer;

CREATE TABLE sat_dispensed_performer (
	dispensed_performer_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	function_code VARCHAR,
	function_concept_id BIGINT,
	function_raw_code VARCHAR,
	function_raw_concept_id BIGINT,
	party_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_performer ADD CONSTRAINT pk_meddispensed_satdispensedperformer_dispensedperformerid PRIMARY KEY (dispensed_performer_id);
ALTER TABLE sat_dispensed_performer ADD CONSTRAINT fk_meddispensed_satdispensedperformer_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_performer.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_performer.dispensed_performer_id IS 'Dispensed Performer Identity';
COMMENT ON COLUMN sat_dispensed_performer.function_code IS 'What the performer did. E.g. dataenterer, packager, checker, finalchecker, counsellor. See https://build.fhir.org/valueset-medicationdispense-performer-function.html';
COMMENT ON COLUMN sat_dispensed_performer.party_id IS 'Who Performed the Dispense Event. E.g. Practitioner, PractitionerRole, Organization, Patient, Device, RelatedPerson, CareTeam';
COMMENT ON COLUMN sat_dispensed_performer.rank IS 'The recommended order the Dispense Function Codes should be presented';



-- -----------------------------------------------------------
-- create sat_dispensed_receiver table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_receiver;

CREATE TABLE sat_dispensed_receiver (
	dispensed_receiver_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	party_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_receiver ADD CONSTRAINT pk_meddispensed_satdispensedreceiver_dispensedreceiverid PRIMARY KEY (dispensed_receiver_id);
ALTER TABLE sat_dispensed_receiver ADD CONSTRAINT fk_meddispensed_satdispensedreceiver_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_receiver.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_receiver.dispensed_receiver_id IS 'Dispensed Receiver Identity';
COMMENT ON COLUMN sat_dispensed_receiver.party_id IS 'The Party who collected the medication or where the medication was delivered. E.g. Patient, Practitioner, RelatedPerson, Location, PractitionerRole. See https://build.fhir.org/medicationdispense-definitions.html#MedicationDispense.receiver';
COMMENT ON COLUMN sat_dispensed_receiver.rank IS 'The recommended order the Dispensed Receiver Entries should be presented';



-- -----------------------------------------------------------
-- create sat_dispensed_substitution_reason table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_substitution_reason;

CREATE TABLE sat_dispensed_substitution_reason (
	dispensed_substitution_reason_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	reason_code VARCHAR,
	reason_concept_id BIGINT,
	reason_raw_code VARCHAR,
	reason_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_substitution_reason ADD CONSTRAINT pk_meddispensed_satdispensedsubstreason_dispensedsubstreasonid PRIMARY KEY (dispensed_substitution_reason_id);
ALTER TABLE sat_dispensed_substitution_reason ADD CONSTRAINT fk_meddispensed_satdispensedsubstreason_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_substitution_reason.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_substitution_reason.dispensed_substitution_reason_id IS 'Dispensed Substitution Reason Identity';
COMMENT ON COLUMN sat_dispensed_substitution_reason.reason_code IS 'Why was substitution made. E.g. CT (continuing therapy), FP (formulary policy), OS (out of stock), RR (regulatory requirement). See https://terminology.hl7.org/5.4.0/ValueSet-v3-SubstanceAdminSubstitutionReason.html';
COMMENT ON COLUMN sat_dispensed_substitution_reason.rank IS 'The recommended order the Medication Dispensed Substitution Reason Codes should be presented';



-- -----------------------------------------------------------
-- create sat_dispensed_supporting_information table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dispensed_supporting_information;

CREATE TABLE sat_dispensed_supporting_information (
	dispensed_supporting_information_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_dispensed_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	identifier_type VARCHAR,
	identifier BIGINT,
	reference VARCHAR,
	notes TEXT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dispensed_supporting_information ADD CONSTRAINT pk_meddispensed_satdispensedsuppinfo_dispensedsuppinfoid PRIMARY KEY (dispensed_supporting_information_id);
ALTER TABLE sat_dispensed_supporting_information ADD CONSTRAINT fk_meddispensed_satdispensedsuppinfo_hubmeddispensed_meddispensedid FOREIGN KEY (medication_dispensed_id) REFERENCES hub_medication_dispensed(medication_dispensed_id);

COMMENT ON COLUMN sat_dispensed_supporting_information.medication_dispensed_id IS 'Medication Dispensed Identity';
COMMENT ON COLUMN sat_dispensed_supporting_information.dispensed_supporting_information_id IS 'Dispensed Supporting Information Identity';
COMMENT ON COLUMN sat_dispensed_supporting_information.identifier_type IS 'Type the reference refers to (e.g. "Patient") - must be a resource in resources. See https://hl7.org/fhir/R5/valueset-resource-types.html';
COMMENT ON COLUMN sat_dispensed_supporting_information.identifier IS 'Logical reference, when literal reference is not known';
COMMENT ON COLUMN sat_dispensed_supporting_information.reference IS 'Literal reference, Relative, internal or absolute URL';
COMMENT ON COLUMN sat_dispensed_supporting_information.notes IS 'Human Actor entered supporting information';
COMMENT ON COLUMN sat_dispensed_supporting_information.rank IS 'The recommended order the Dispense Supporting Information Entries should be presented';




-- -----------------------------------------------------------
-- create view_medication_dispensed view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_medication_dispensed AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_dispensed_id,
	h.medication_dispensed_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_medication_dispensed_id,
  s.medication_prescription_id,
  s.medication_request_id,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.not_performed_reason_code,
  s.not_performed_reason_concept_id,
  s.not_performed_reason_raw_code,
  s.not_performed_reason_raw_concept_id,
  s.status_changed,
  s.medication_code,
  s.medication_concept_id,
  s.medication_raw_code,
  s.medication_raw_concept_id,
  s.location_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.quantity,
  s.days_supply,
  s.recorded,
  s.when_prepared,
  s.when_handed_over,
  s.destination_location_id,
  s.rendered_dosage_instructions,
  s.was_substituted,
  s.substitution_type_code,
  s.substitution_type_concept_id,
  s.substitution_type_raw_code,
  s.substitution_type_raw_concept_id,
  s.substitution_responsible_party_id,
  s.record_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_dispensed.hub_medication_dispensed h
    JOIN budd.qdp_medication_dispensed.sat_medication_dispensed s
      ON h.medication_dispensed_id = s.medication_dispensed_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_medication_dispensed;


-- -----------------------------------------------------------
-- create view_dispensed_categories view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_dispensed_categories AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_dispensed_id,
	h.medication_dispensed_entity_id,
  s.status_code,
  s.medication_code,
	dc.dispensed_category_id,
	dc.category_code,
	dc.category_concept_id,
	dc.category_raw_code,
	dc.category_raw_concept_id,
	dc.rank AS category_rank,
	dc.period_start AS category_period_start,
	dc.period_end AS category_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_dispensed.hub_medication_dispensed h
    JOIN budd.qdp_medication_dispensed.sat_medication_dispensed s
      ON h.medication_dispensed_id = s.medication_dispensed_id
    JOIN budd.qdp_medication_dispensed.sat_dispensed_category dc
      ON h.medication_dispensed_id = dc.medication_dispensed_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_dispensed_categories;


-- -----------------------------------------------------------
-- create view_dispensed_performers view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_dispensed_performers AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_dispensed_id,
	h.medication_dispensed_entity_id,
  s.status_code,
  s.medication_code,
	dp.dispensed_performer_id,
	dp.function_code,
	dp.function_concept_id,
	dp.function_raw_code,
	dp.function_raw_concept_id,
	dp.party_id AS performer_party_id,
	dp.rank AS performer_rank,
	dp.period_start AS performer_period_start,
	dp.period_end AS performer_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_dispensed.hub_medication_dispensed h
    JOIN budd.qdp_medication_dispensed.sat_medication_dispensed s
      ON h.medication_dispensed_id = s.medication_dispensed_id
    JOIN budd.qdp_medication_dispensed.sat_dispensed_performer dp
      ON h.medication_dispensed_id = dp.medication_dispensed_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_dispensed_performers;


-- -----------------------------------------------------------
-- create view_dispensed_substitution_reasons view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_dispensed_substitution_reasons AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_dispensed_id,
	h.medication_dispensed_entity_id,
  s.status_code,
  s.medication_code,
  s.was_substituted,
  s.substitution_type_code,
	sr.dispensed_substitution_reason_id,
	sr.reason_code,
	sr.reason_concept_id,
	sr.reason_raw_code,
	sr.reason_raw_concept_id,
	sr.rank AS reason_rank,
	sr.period_start AS reason_period_start,
	sr.period_end AS reason_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_dispensed.hub_medication_dispensed h
    JOIN budd.qdp_medication_dispensed.sat_medication_dispensed s
      ON h.medication_dispensed_id = s.medication_dispensed_id
    JOIN budd.qdp_medication_dispensed.sat_dispensed_substitution_reason sr
      ON h.medication_dispensed_id = sr.medication_dispensed_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_dispensed_substitution_reasons;


/**

**/
